# Загрузка данных

In [ ]:
# Библиотеки для работы с данными
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import GroupKFold, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer
import shap
from pathlib import Path

In [ ]:
df_m1 = pd.read_csv ('data_for_models/df_m1_ols.csv', low_memory=False)

In [ ]:
df_m1.info()

# Подготовка данных

In [ ]:
features = ['role_name', 'experience_ord', 'schedule_id', 'employment_id']
group_col = 'region_name'
target = 'salary_from_log'


# Дедупликация ОДИНАКОВО для всех веток: структура + таргет + регион.
# Смысл: одна и та же структура и зарплата в РАЗНЫХ регионах — разные строки.
dedup_cols = features + [target, group_col]
df_m1_base = df_m1[dedup_cols].drop_duplicates(subset=dedup_cols).reset_index(drop=True)
df_m1_cv = df_m1_base

print(f"Каноническая выборка M1 (дедупликация по {dedup_cols}): {df_m1_base.shape[0]} строк")
print(df_m1_base[group_col].value_counts().describe())

# OLS без CV (robust SE) + визуализация влияния категорий

In [ ]:
# Формула для OLS (robust SE)
formula = "salary_from_log ~ C(role_name) + experience_ord + C(schedule_id) + C(employment_id)"
ols_model = smf.ols(formula=formula, data=df_m1_base).fit(cov_type="HC3")
print(ols_model.summary())

# Коэффициенты и процентное влияние
params = ols_model.params
conf = ols_model.conf_int()

results_df = pd.DataFrame({
    'feature': params.index,
    'coef': params.values,
    'ci_lower': conf[0],
    'ci_upper': conf[1]
}).reset_index(drop=True)

# Убираем интерсепт
results_df = results_df[results_df['feature'] != 'Intercept']

# Переводим коэффициенты в проценты изменения зарплаты
results_df['pct_effect'] = (np.expm1(results_df['coef'])) * 100
results_df['pct_lower'] = (np.expm1(results_df['ci_lower'])) * 100
results_df['pct_upper'] = (np.expm1(results_df['ci_upper'])) * 100

# Разделяем признаки по группам
role_effects = results_df[results_df['feature'].str.contains('role_name')].sort_values('pct_effect')
schedule_effects = results_df[results_df['feature'].str.contains('schedule_id')].sort_values('pct_effect')
employment_effects = results_df[results_df['feature'].str.contains('employment_id')].sort_values('pct_effect')

# Визуализация влияния профессиональной роли
plt.figure(figsize=(10,6))
sns.barplot(data=role_effects, x='pct_effect', y='feature', color='skyblue')
plt.axvline(0, color='black', linestyle='--')
plt.xlabel("Изменение зарплаты (%) относительно базовой категории")
plt.ylabel("Профессиональная роль")
plt.title("Влияние профессиональной роли на зарплату (OLS M1)")
plt.tight_layout()
plt.show()

# Визуализация влияния графика работы 
plt.figure(figsize=(8,4))
sns.barplot(data=schedule_effects, x='pct_effect', y='feature', color='lightgreen')
plt.axvline(0, color='black', linestyle='--')
plt.xlabel("Изменение зарплаты (%) относительно базовой категории")
plt.ylabel("Тип графика работы")
plt.title("Влияние графика работы на зарплату (OLS M1)")
plt.tight_layout()
plt.show()

# Визуализация влияния типа занятости
plt.figure(figsize=(8,4))
sns.barplot(data=employment_effects, x='pct_effect', y='feature', color='salmon')
plt.axvline(0, color='black', linestyle='--')
plt.xlabel("Изменение зарплаты (%) относительно базовой категории")
plt.ylabel("Тип занятости")
plt.title("Влияние типа занятости на зарплату (OLS M1)")
plt.tight_layout()
plt.show()

# Распределение остатков
residuals = ols_model.resid
plt.figure(figsize=(8,5))
sns.histplot(residuals, bins=50, kde=True)
plt.title("Распределение остатков OLS без CV")
plt.xlabel("Остатки")
plt.ylabel("Частота")
plt.tight_layout()
plt.show()

# Метрики на обучении
rmse_ols_simple = np.sqrt(mean_squared_error(df_m1_base[target], ols_model.fittedvalues))
mae_ols_simple = mean_absolute_error(df_m1_base[target], ols_model.fittedvalues)
r2_ols_simple = r2_score(df_m1_base[target], ols_model.fittedvalues)

print(f"OLS без CV: RMSE={rmse_ols_simple:.4f}, MAE={mae_ols_simple:.4f}, R2={r2_ols_simple:.4f}")

# OLS с GroupKFold

In [ ]:
categorical_features = ["role_name", "schedule_id", "employment_id"]
numeric_features = ["experience_ord"]
def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
            ("num", StandardScaler(), numeric_features),
        ]
    )
N_FOLDS = 5
gkf = GroupKFold(n_splits=N_FOLDS)

In [ ]:
pipeline_ols = Pipeline(
    [
        ("preprocessor", make_preprocessor()),
        ("model", LinearRegression()),
    ]
)
rmse_list, mae_list, r2_list = [], [], []
for train_idx, test_idx in gkf.split(df_m1_base, groups=df_m1_base[group_col]):
    X_train = df_m1_base.iloc[train_idx][features]
    y_train = df_m1_base.iloc[train_idx][target]
    X_test = df_m1_base.iloc[test_idx][features]
    y_test = df_m1_base.iloc[test_idx][target]
    pipeline_ols.fit(X_train, y_train)
    preds = pipeline_ols.predict(X_test)
    rmse_list.append(np.sqrt(mean_squared_error(y_test, preds)))
    mae_list.append(mean_absolute_error(y_test, preds))
    r2_list.append(r2_score(y_test, preds))
cv_rmse_ols_mean, cv_rmse_ols_std = float(np.mean(rmse_list)), float(np.std(rmse_list))
cv_mae_ols_mean, cv_mae_ols_std = float(np.mean(mae_list)), float(np.std(mae_list))
cv_r2_ols_mean, cv_r2_ols_std = float(np.mean(r2_list)), float(np.std(r2_list))
fold_metrics_ols = pd.DataFrame(
    {"fold": np.arange(1, len(rmse_list) + 1), "RMSE": rmse_list, "MAE": mae_list, "R2": r2_list}
)
print(
    f"OLS(GroupKFold): RMSE {cv_rmse_ols_mean:.4f} ± {cv_rmse_ols_std:.4f}, "
    f"MAE {cv_mae_ols_mean:.4f} ± {cv_mae_ols_std:.4f}, "
    f"R2 {cv_r2_ols_mean:.4f} ± {cv_r2_ols_std:.4f}"
)
display(fold_metrics_ols)


# Ridge Regression с GroupKFold и подбором alpha

In [ ]:
pipeline_ridge = Pipeline(
    [
        ("preprocessor", make_preprocessor()),
        ("model", Ridge()),
    ]
)
param_grid = {"model__alpha": [0.01, 0.1, 1, 10, 100]}
grid_search = GridSearchCV(
    pipeline_ridge,
    param_grid,
    cv=gkf,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
grid_search.fit(df_m1_base[features], df_m1_base[target], groups=df_m1_base[group_col])
best_model = grid_search.best_estimator_
best_alpha = grid_search.best_params_["model__alpha"]
rmse_folds = -cross_val_score(
    best_model,
    df_m1_base[features],
    df_m1_base[target],
    cv=gkf,
    groups=df_m1_base[group_col],
    scoring="neg_root_mean_squared_error",
)
mae_folds = -cross_val_score(
    best_model,
    df_m1_base[features],
    df_m1_base[target],
    cv=gkf,
    groups=df_m1_base[group_col],
    scoring="neg_mean_absolute_error",
)
r2_folds = cross_val_score(
    best_model,
    df_m1_base[features],
    df_m1_base[target],
    cv=gkf,
    groups=df_m1_base[group_col],
    scoring="r2",
)
cv_rmse_ridge_mean, cv_rmse_ridge_std = float(np.mean(rmse_folds)), float(np.std(rmse_folds))
cv_mae_ridge_mean, cv_mae_ridge_std = float(np.mean(mae_folds)), float(np.std(mae_folds))
cv_r2_ridge_mean, cv_r2_ridge_std = float(np.mean(r2_folds)), float(np.std(r2_folds))
fold_metrics_ridge = pd.DataFrame(
    {"fold": np.arange(1, len(rmse_folds) + 1), "RMSE": rmse_folds, "MAE": mae_folds, "R2": r2_folds}
)
print(f"Ridge best alpha: {best_alpha}")
print(
    f"Ridge(GroupKFold): RMSE {cv_rmse_ridge_mean:.4f} ± {cv_rmse_ridge_std:.4f}, "
    f"MAE {cv_mae_ridge_mean:.4f} ± {cv_mae_ridge_std:.4f}, "
    f"R2 {cv_r2_ridge_mean:.4f} ± {cv_r2_ridge_std:.4f}"
)
display(fold_metrics_ridge)

In [ ]:
# 1) Сравнение сетки alpha по RMSE
gs_res = pd.DataFrame(grid_search.cv_results_)[
    ["param_model__alpha", "mean_test_score", "std_test_score", "rank_test_score"]
].copy()
gs_res["mean_RMSE"] = -gs_res["mean_test_score"]
gs_res["std_RMSE"] = gs_res["std_test_score"]
display(gs_res.sort_values("param_model__alpha"))

In [ ]:
# 2) Прямое сравнение предсказаний OLS vs Ridge на одном фолде
from sklearn.base import clone
train_idx, test_idx = next(gkf.split(df_m1_base, groups=df_m1_base[group_col]))
X_train = df_m1_base.iloc[train_idx][features]
y_train = df_m1_base.iloc[train_idx][target]
X_test = df_m1_base.iloc[test_idx][features]
ols_fold = clone(pipeline_ols).fit(X_train, y_train)
ridge_fold = clone(best_model).fit(X_train, y_train)
pred_ols = ols_fold.predict(X_test)
pred_ridge = ridge_fold.predict(X_test)
print("Mean abs diff preds:", np.mean(np.abs(pred_ols - pred_ridge)))
print("Max abs diff preds:", np.max(np.abs(pred_ols - pred_ridge)))
print("Corr preds:", np.corrcoef(pred_ols, pred_ridge)[0, 1])

In [ ]:
# 3) Насколько Ridge действительно "сжал" коэффициенты
coef_ols = ols_fold.named_steps["model"].coef_
coef_ridge = ridge_fold.named_steps["model"].coef_
print("L2 norm OLS:", np.linalg.norm(coef_ols, 2))
print("L2 norm Ridge:", np.linalg.norm(coef_ridge, 2))
print("L2 ratio Ridge/OLS:", np.linalg.norm(coef_ridge, 2) / np.linalg.norm(coef_ols, 2))

Диагностика по сетке alpha показывает плато качества в диапазоне 0.01–10, а прямое сравнение предсказаний (corr≈0.999999) подтверждает, что Ridge и OLS дают практически одинаковые прогнозы. При этом Ridge выполняет умеренное сжатие коэффициентов (L2 ratio≈0.93), но его вклад в out-of-sample метрики минимален, что соответствует слабой чувствительности задачи к регуляризации на текущем наборе структурных признаков. Следовательно, отсутствие заметного прироста Ridge относительно OLS интерпретируется как содержательный результат данных, а не как техническая ошибка реализации.

# Сравнение метрик

In [ ]:
# Train / in-sample (statsmodels OLS на полной df_m1_base)
train_metrics = pd.DataFrame(
    [
        {
            "Model": "OLS (statsmodels, HC3, in-sample)",
            "Train_RMSE": rmse_ols_simple,
            "Train_MAE": mae_ols_simple,
            "Train_R2": r2_ols_simple,
        }
    ]
)
# --- CV (единый формат, как на M2) ---
cv_metrics = pd.DataFrame(
    [
        {
            "Model": "OLS (sklearn pipeline, GroupKFold)",
            "CV_RMSE_mean": cv_rmse_ols_mean,
            "CV_RMSE_std": cv_rmse_ols_std,
            "CV_MAE_mean": cv_mae_ols_mean,
            "CV_MAE_std": cv_mae_ols_std,
            "CV_R2_mean": cv_r2_ols_mean,
            "CV_R2_std": cv_r2_ols_std,
            "n_folds": N_FOLDS,
            "Best_alpha": np.nan,
        },
        {
            "Model": "Ridge (sklearn pipeline, GroupKFold)",
            "CV_RMSE_mean": cv_rmse_ridge_mean,
            "CV_RMSE_std": cv_rmse_ridge_std,
            "CV_MAE_mean": cv_mae_ridge_mean,
            "CV_MAE_std": cv_mae_ridge_std,
            "CV_R2_mean": cv_r2_ridge_mean,
            "CV_R2_std": cv_r2_ridge_std,
            "n_folds": N_FOLDS,
            "Best_alpha": best_alpha,
        },
    ]
)
baseline_rmse = cv_metrics.loc[
    cv_metrics["Model"].str.startswith("OLS"), "CV_RMSE_mean"
].values[0]
baseline_r2 = cv_metrics.loc[
    cv_metrics["Model"].str.startswith("OLS"), "CV_R2_mean"
].values[0]
cv_metrics["RMSE_improvement_%"] = (
    baseline_rmse - cv_metrics["CV_RMSE_mean"]
) / baseline_rmse * 100
cv_metrics["R2_gain"] = cv_metrics["CV_R2_mean"] - baseline_r2
print("=== Train / in-sample (не смешивать с CV) ===")
display(train_metrics.round(4))
print("=== CV metrics ===")
display(cv_metrics.round(6))
plot_df = cv_metrics.set_index("Model")[
    ["CV_RMSE_mean", "CV_MAE_mean", "CV_R2_mean"]
].rename(
    columns={
        "CV_RMSE_mean": "CV RMSE (mean)",
        "CV_MAE_mean": "CV MAE (mean)",
        "CV_R2_mean": "CV R2 (mean)",
    }
)
plot_df.plot(kind="bar", figsize=(10, 5))
plt.title("M1: CV metrics (GroupKFold) — mean only")
plt.ylabel("Metric value")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
#Сохраним метрики
metrics_dir = Path("data_for_models")
metrics_dir.mkdir(parents=True, exist_ok=True)
# Сохраняем только CV-метрики (без in-sample)
m1_cv_metrics = cv_metrics.copy()
preferred_cols = [
    "Model",
    "CV_RMSE_mean", "CV_RMSE_std",
    "CV_MAE_mean", "CV_MAE_std",
    "CV_R2_mean", "CV_R2_std",
    "n_folds",
    "RMSE_improvement_%", "R2_gain"
]
m1_cv_metrics = m1_cv_metrics[[c for c in preferred_cols if c in m1_cv_metrics.columns]]
m1_cv_metrics_path = metrics_dir / "m1_cv_metrics.csv"
m1_cv_metrics.to_csv(m1_cv_metrics_path, index=False, encoding="utf-8-sig")
print(f"Saved: {m1_cv_metrics_path}")
display(m1_cv_metrics.round(6))

# Feature Importance и SHAP для Ridge

In [ ]:
# Предварительно преобразуем X через preprocessor
X_ridge = best_model.named_steps["preprocessor"].transform(df_m1_base[features])
feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()

# Создаем explainer
explainer = shap.LinearExplainer(
    best_model.named_steps['model'],
    X_ridge,
    feature_perturbation="interventional"
)

# Вычисляем SHAP
shap_values = explainer.shap_values(X_ridge)

# Считаем mean absolute SHAP
shap_df = pd.DataFrame(
    np.abs(shap_values).mean(axis=0),
    index=feature_names,
    columns=['mean_abs_shap']
).sort_values('mean_abs_shap', ascending=False)

# Строим график
plt.figure(figsize=(10,6))
sns.barplot(
    x='mean_abs_shap',
    y=shap_df.index,
    data=shap_df.reset_index(),
    palette='coolwarm'
)
plt.title("Ridge SHAP - Mean |SHAP value|")
plt.show()

Сделаем интерпретацию в рублях/процентах для перевода эффектов из log-шкалы в прикладную шкалу зарплаты на референсном уровне (медиана). Для SHAP это аппроксимация типичного масштаба вклада, поскольку базовая аддитивность SHAP задана в log-пространстве.

In [ ]:
# Интерпретация эффектов в процентах/рублях 
# Модель обучена на salary_from_log, поэтому базовая интерпретация — в log-шкале.
# Ниже даем удобный перевод в % и условные рубли на "типичной" зарплате.
# 1) Референсный уровень зарплаты (медиана по выборке в исходной шкале)
salary_ref_rub = float(np.expm1(df_m1_base[target]).median())
print(f"Reference salary (median, RUB): {salary_ref_rub:,.0f}".replace(",", " "))
# 2) OLS: перевод коэффициентов в % уже есть в results_df['pct_effect']
# Добавим эквивалент в рублях на референсной зарплате
ols_interp = results_df.copy()
ols_interp["rub_effect_at_ref"] = salary_ref_rub * (np.expm1(ols_interp["coef"]))
ols_interp["rub_lower_at_ref"] = salary_ref_rub * (np.expm1(ols_interp["ci_lower"]))
ols_interp["rub_upper_at_ref"] = salary_ref_rub * (np.expm1(ols_interp["ci_upper"]))
# Покажем наиболее сильные эффекты по модулю
ols_top = (
    ols_interp.assign(abs_pct=ols_interp["pct_effect"].abs())
    .sort_values("abs_pct", ascending=False)
    .head(12)[
        [
            "feature",
            "coef",
            "pct_effect",
            "pct_lower",
            "pct_upper",
            "rub_effect_at_ref",
            "rub_lower_at_ref",
            "rub_upper_at_ref",
        ]
    ]
)
display(ols_top.round(2))
# 3) SHAP (Ridge): вклад в log-шкале  приблизительный вклад в % и рублях
#рублёвая интерпретация SHAP это аппроксимация, базовая аддитивность SHAP остаётся в log-шкале.
shap_arr = np.array(shap_values)
if shap_arr.ndim == 3:  # на случай необычного формата
    shap_arr = shap_arr[0]
mean_abs_shap_log = np.abs(shap_arr).mean(axis=0)
shap_interp = pd.DataFrame(
    {
        "feature": feature_names,
        "mean_abs_shap_log": mean_abs_shap_log,
    }
).sort_values("mean_abs_shap_log", ascending=False)
# Приближенный "типичный" вклад:
# delta_% ≈ (exp(|SHAP|)-1)*100
# delta_rub ≈ salary_ref * (exp(|SHAP|)-1)
shap_interp["approx_typical_pct"] = np.expm1(shap_interp["mean_abs_shap_log"]) * 100
shap_interp["approx_typical_rub"] = salary_ref_rub * np.expm1(shap_interp["mean_abs_shap_log"])
print("\nTop SHAP features with approximate effect scale:")
display(shap_interp.head(15).round(3))
